# Product Category Classification using Machine Learning

This project predicts the category of a product based on its name and description using NLP and Machine Learning.

# Import Libraries

In [2]:
import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# Load Dataset

In [3]:
import pandas as pd

df = pd.read_csv("amazon.csv")

df.columns = df.columns.str.strip().str.lower()

df.head()

,product_id,product_name,category,discounted_price,actual_price,discount_percentage,rating,rating_count,about_product,user_id,user_name,review_id,review_title,review_content,img_link,product_link
0,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories|Accessories&Peripherals|...,₹399,"₹1,099",64%,4.2,"24,269",High Compatibility : Compatible With iPhone 12...,"AG3D6O4STAQKAY2UVGEUV46KN35Q,AHMY5CWJMMK5BJRBB...","Manav,Adarsh gupta,Sundeep,S.Sayeed Ahmed,jasp...","R3HXWT0LRP0NMF,R2AJM3LFTLZHFO,R6AQJGUP6P86,R1K...","Satisfied,Charging is really fast,Value for mo...",Looks durable Charging is fine tooNo complains...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Wayona-Braided-WN3LG1-Sy...
1,B098NS6PVG,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,Computers&Accessories|Accessories&Peripherals|...,₹199,₹349,43%,4.0,"43,994","Compatible with all Type C enabled devices, be...","AECPFYFQVRUWC3KGNLJIOREFP5LQ,AGYYVPDD7YG7FYNBX...","ArdKn,Nirbhay kumar,Sagar Viswanathan,Asp,Plac...","RGIQEG07R9HS2,R1SMWZQ86XIN8U,R2J3Y1WL29GWDE,RY...","A Good Braided Cable for Your Type C Device,Go...",I ordered this cable to connect my phone to An...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Ambrane-Unbreakable-Char...
2,B096MSW6CT,Sounce Fast Phone Charging Cable & Data Sync U...,Computers&Accessories|Accessories&Peripherals|...,₹199,"₹1,899",90%,3.9,"7,928",【 Fast Charger& Data Sync】-With built-in safet...,"AGU3BBQ2V2DDAMOAKGFAWDDQ6QHA,AESFLDV2PT363T2AQ...","Kunal,Himanshu,viswanath,sai niharka,saqib mal...","R3J3EQQ9TZI5ZJ,R3E7WBGK7ID0KV,RWU79XKQ6I1QF,R2...","Good speed for earlier versions,Good Product,W...","Not quite durable and sturdy,https://m.media-a...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Sounce-iPhone-Charging-C...
3,B08HDJ86NZ,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...,Computers&Accessories|Accessories&Peripherals|...,₹329,₹699,53%,4.2,"94,363",The boAt Deuce USB 300 2 in 1 cable is compati...,"AEWAZDZZJLQUYVOVGBEUKSLXHQ5A,AG5HTSFRRE6NL3M5S...","Omkar dhale,JD,HEMALATHA,Ajwadh a.,amar singh ...","R3EEUZKKK9J36I,R3HJVYCLYOY554,REDECAZ7AMPQC,R1...","Good product,Good one,Nice,Really nice product...","Good product,long wire,Charges good,Nice,I bou...",https://m.media-amazon.com/images/I/41V5FtEWPk...,https://www.amazon.in/Deuce-300-Resistant-Tang...
4,B08CF3B7N1,Portronics Konnect L 1.2M Fast Charging 3A 8 P...,Computers&Accessories|Accessories&Peripherals|...,₹154,₹399,61%,4.2,"16,905",[CHARGE & SYNC FUNCTION]- This cable comes wit...,"AE3Q6KSUK5P75D5HFYHCRAOLODSA,AFUGIFH5ZAFXRDSZH...","rahuls6099,Swasat Borah,Ajay Wadke,Pranali,RVK...","R1BP4L2HH9TFUP,R16PVJEXKV6QZS,R2UPDB81N66T4P,R...","As good as original,Decent,Good one for second...","Bought this instead of original apple, does th...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Portronics-Konnect-POR-1...


# Select Required Columns

In [4]:
df = df[['product_name', 'about_product', 'category']].copy()
df.head()

,product_name,about_product,category
0,Wayona Nylon Braided USB to Lightning Fast Cha...,High Compatibility : Compatible With iPhone 12...,Computers&Accessories|Accessories&Peripherals|...
1,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,"Compatible with all Type C enabled devices, be...",Computers&Accessories|Accessories&Peripherals|...
2,Sounce Fast Phone Charging Cable & Data Sync U...,【 Fast Charger& Data Sync】-With built-in safet...,Computers&Accessories|Accessories&Peripherals|...
3,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...,The boAt Deuce USB 300 2 in 1 cable is compati...,Computers&Accessories|Accessories&Peripherals|...
4,Portronics Konnect L 1.2M Fast Charging 3A 8 P...,[CHARGE & SYNC FUNCTION]- This cable comes wit...,Computers&Accessories|Accessories&Peripherals|...


# Combine Text

In [5]:
df['text'] = df['product_name'] + " " + df['about_product']

df = df[['text', 'category']].copy()
df.columns = ['text', 'label']

df.head()

,text,label
0,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories|Accessories&Peripherals|...
1,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,Computers&Accessories|Accessories&Peripherals|...
2,Sounce Fast Phone Charging Cable & Data Sync U...,Computers&Accessories|Accessories&Peripherals|...
3,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...,Computers&Accessories|Accessories&Peripherals|...
4,Portronics Konnect L 1.2M Fast Charging 3A 8 P...,Computers&Accessories|Accessories&Peripherals|...


# Text Cleaning

In [6]:
import re

df['text'] = df['text'].str.lower()
df['text'] = df['text'].apply(lambda x: re.sub(r'[^a-zA-Z0-9 ]', '', x))
df['text'] = df['text'].apply(lambda x: " ".join(x.split()))

# Remove Rare Categories

In [7]:
counts = df['label'].value_counts()

valid_labels = counts[counts >= 50].index

df = df[df['label'].isin(valid_labels)]

# Feature Engineering (TF-IDF)

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(df['text'])
y = df['label']

# Train-Test Split

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Model Training

In [10]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X_train, y_train)

MultinomialNB()

# Evaluation

In [11]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))

Accuracy: 0.8282828282828283
                                                                                   precision    recall  f1-score   support

Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables       0.72      1.00      0.84        44
                     Electronics|Headphones,Earbuds&Accessories|Headphones|In-Ear       1.00      0.20      0.33        20
                    Electronics|HomeTheater,TV&Video|Televisions|SmartTelevisions       1.00      1.00      1.00        11
             Electronics|Mobiles&Accessories|Smartphones&BasicMobiles|Smartphones       1.00      1.00      1.00        12
                                      Electronics|WearableTechnology|SmartWatches       1.00      0.92      0.96        12

                                                                         accuracy                           0.83        99
                                                                        macro avg       0.94      0.82      

# Top-3 Prediction

In [12]:
import numpy as np

probs = model.predict_proba(X_test)

top3 = np.argsort(probs, axis=1)[:, -3:]

print(top3[:5])

[[3 4 0]
 [2 4 0]
 [1 4 0]
 [4 0 3]
 [2 4 0]]


# Custom Input Testing

In [13]:
sample = ["iphone charger fast charging cable"]

sample_vec = vectorizer.transform(sample)

probs = model.predict_proba(sample_vec)

top3 = np.argsort(probs)[0][-3:]

print("Top 3 categories:", top3)
print("Confidence:", probs[0][top3])

Top 3 categories: [4 3 0]
Confidence: [0.00490295 0.00596487 0.98424499]


In [16]:
import numpy as np

sample = ["wireless bluetooth headphones with noise cancellation"]

sample_vec = vectorizer.transform(sample)
probs = model.predict_proba(sample_vec)

top3_idx = np.argsort(probs)[0][-3:][::-1]  # reverse for highest first
labels = model.classes_

print("🔍 Input:", sample[0])
print("\n🎯 Top 3 Predictions:")

for idx in top3_idx:
    label = labels[idx].split('|')[0]
    score = probs[0][idx]
    print(f"👉 {label} → {score:.2f}")

🔍 Input: wireless bluetooth headphones with noise cancellation

🎯 Top 3 Predictions:
👉 Computers&Accessories → 0.36
👉 Electronics → 0.22
👉 Electronics → 0.20


# Model Accuracy

In [15]:
print("🔥 Model Accuracy:", accuracy_score(y_test, y_pred))
print("\n📊 Classification Report:\n")
print(classification_report(y_test, y_pred, zero_division=0))

🔥 Model Accuracy: 0.8282828282828283

📊 Classification Report:

                                                                                   precision    recall  f1-score   support

Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables       0.72      1.00      0.84        44
                     Electronics|Headphones,Earbuds&Accessories|Headphones|In-Ear       1.00      0.20      0.33        20
                    Electronics|HomeTheater,TV&Video|Televisions|SmartTelevisions       1.00      1.00      1.00        11
             Electronics|Mobiles&Accessories|Smartphones&BasicMobiles|Smartphones       1.00      1.00      1.00        12
                                      Electronics|WearableTechnology|SmartWatches       1.00      0.92      0.96        12

                                                                         accuracy                           0.83        99
                                                                        m

# Conclusion

- The model achieved an accuracy of ~82%.
- TF-IDF helped convert text into numerical features.
- Naive Bayes performed well for text classification.
- The model can predict top categories for new product inputs.

Future improvements:
- Try Logistic Regression
- Use Deep Learning (LSTM/BERT)
- Improve data cleaning